# MLOps: Ciclo de Vida de un Modelo de Machine Learning
## Ejemplo práctico: Predicción de Precio de Viviendas en California

Bienvenido a esta serie de notebooks que demuestran el ciclo completo de MLOps
usando un ejemplo real: predecir el precio de viviendas en California.

## Con que trabajamos?

**Dataset:** California Housing (incluido en scikit-learn)
- 20,640 viviendas del censo de California de 1990
- **Objetivo:** predecir el precio mediano de viviendas por bloque
- **Variables:** ingresos, antigüedad, habitaciones, ubicación geográfica

**Criterios de éxito:**
- RMSE < 0.5 (escala normalizada)
- R2 > 0.80

## El Ciclo de Vida MLOps

```
+--------------------------------------------------------------------+
|                      CICLO DE VIDA MLOps                           |
|                                                                    |
|  00_introduccion          -> Este notebook: vision general         |
|  01_eda_exploratorio      -> Analisis Exploratorio de Datos        |
|  02_ingenieria_features   -> Feature Engineering + StandardScaler  |
|  03_entrenamiento_mlflow  -> Entrenamiento + Tracking MLflow       |
|  04_seleccion_modelo      -> Cross-Validation + Curvas aprendizaje |
|  05_evaluacion_final      -> Gate de calidad + Model Registry      |
|  06_despliegue_api        -> Docker + FastAPI + Web UI             |
|  07_monitoreo_reentren.   -> PSI, drift, GitHub Actions            |
|  08_automl                -> Optuna: busqueda automatica hiperpar. |
|  09_explicabilidad_shap   -> SHAP values, beeswarm, waterfall      |
|  10_validacion_datos      -> Pandera: schema contracts             |
|  11_ab_testing            -> A/B Testing + aprobacion manual       |
|  12_model_card            -> Documentacion: metricas, limites,etica|
|  13_inferencia_batch      -> Predicciones masivas + scheduling     |
+--------------------------------------------------------------------+
```

Cada notebook es **prerequisito** del siguiente. Ejecútalos en orden.

In [1]:
# Verificar que el entorno está correctamente configurado
import sys
from pathlib import Path

ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

print("=" * 55)
print("  VERIFICACION DEL ENTORNO")
print("=" * 55)

# Python version
print(f"\n  Python: {sys.version.split()[0]}")

# Librerías clave
libs = [
    ('pandas',      'pandas'),
    ('numpy',       'numpy'),
    ('sklearn',     'scikit-learn'),
    ('matplotlib',  'matplotlib'),
    ('seaborn',     'seaborn'),
    ('mlflow',      'mlflow'),
    ('fastapi',     'fastapi'),
    ('optuna',      'optuna'),
    ('shap',        'shap'),
    ('pandera',     'pandera'),
    ('uvicorn',     'uvicorn'),
]
print()
for import_name, display_name in libs:
    try:
        mod = __import__(import_name)
        version = getattr(mod, '__version__', 'OK')
        print(f"  OK  {display_name:<18} {version}")
    except ImportError:
        print(f"  FALTA  {display_name:<18} NO INSTALADO")

# Estructura del proyecto
print(f"\n  ROOT del proyecto: {ROOT}")
carpetas = ['src', 'data', 'experiments', 'notebooks', 'config', 'pipeline', 'tests']
for c in carpetas:
    estado = "OK" if (ROOT / c).exists() else "FALTA"
    print(f"  {estado}  {c}/")

print("\n  Listo! Sigue con 01_eda_exploratorio.ipynb")

  VERIFICACION DEL ENTORNO

  Python: 3.11.5



  OK  pandas             2.2.2
  OK  numpy              1.26.4


  OK  scikit-learn       1.4.2


  OK  matplotlib         3.8.4


  OK  seaborn            0.13.2


  OK  mlflow             3.10.1
  OK  fastapi            0.135.1


  OK  optuna             4.8.0


  OK  shap               0.45.1


  OK  pandera            0.30.0
  OK  uvicorn            0.42.0

  ROOT del proyecto: C:\Users\bk70827\PycharmProjects\mlops-ciclo-vida
  OK  src/
  OK  data/
  OK  experiments/
  OK  notebooks/
  OK  config/
  OK  pipeline/
  OK  tests/

  Listo! Sigue con 01_eda_exploratorio.ipynb


## Stack Tecnologico

| Herramienta | Uso en el proyecto |
|---|---|
| **scikit-learn** | Modelos ML y metricas |
| **pandas / numpy** | Manipulacion de datos |
| **matplotlib / seaborn** | Visualizaciones |
| **MLflow** | Tracking de experimentos + Model Registry |
| **Optuna** | AutoML: busqueda automatica de hiperparametros (TPE) |
| **SHAP** | Explicabilidad: contribucion de cada feature a cada prediccion |
| **Pandera** | Validacion de datos: schema contracts y deteccion de errores |
| **FastAPI + uvicorn** | API REST para servir el modelo en produccion |
| **Docker** | Containerizacion del servicio |
| **GitHub Actions** | CI/CD automatico (4 jobs: ci, monitoreo, train/eval, deploy) |

## Conceptos Clave que Aprenderas

- **Data Leakage**: por que el scaler solo se ajusta con datos de entrenamiento
- **Bias-Variance Tradeoff**: sobreajuste vs. subajuste
- **Cross-Validation**: evaluacion robusta con 5 folds
- **Model Registry**: versionar y promover modelos a produccion con MLflow
- **Training-Serving Parity**: la API aplica las mismas transformaciones que el entrenamiento
- **Gate de Calidad**: el modelo solo va a produccion si supera los umbrales (RMSE < 0.5, R² > 0.80)
- **Data Drift**: deteccion de cambios en la distribucion de los datos con PSI
- **Concept Drift**: degradacion del modelo en produccion (RMSE crece > 10%)
- **AutoML**: busqueda automatica de hiperparametros con Optuna (TPE sampler)
- **Explicabilidad**: SHAP values — cuanto y en que direccion contribuye cada feature
- **Schema Contracts**: Pandera valida que los datos cumplen el esquema esperado antes de usarlos
- **A/B Testing**: prueba controlada entre modelo en produccion y challenger, con analisis estadistico y aprobacion manual
- **Model Card**: documento estructurado de gobernanza — metricas por subgrupo, limitaciones y consideraciones eticas
- **Inferencia Batch**: predicciones masivas programadas con trazabilidad completa y scheduling